<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UNet (traditional)

This notebook will implement the UNet CNN as described in the [original paper](https://arxiv.org/pdf/1505.04597.pdf)

Imports

In [ ]:
import os

import torch

from torch.utils.data import Dataset
from torch import nn
from torchvision.io import read_image
from torchvision import transforms
from torchvision.transforms.functional import crop

import matplotlib.pyplot as plt

import cv2

import numpy as np

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/global_modules.ipynb"

SegmentationDataset class

In [ ]:
class SegmentationDataset(Dataset):
  def __init__(self, img_dir, mask_dir, label_map, transform, device):
    self.img_dir = img_dir
    self.mask_dir = mask_dir
    self.ids = os.listdir(img_dir)
    self.label_map = label_map
    self.transform = transform
    self.device = device

  def __len__(self):
    return len(self.ids)

  def __getitem__(self, idx):
    img_path = os.path.join(self.img_dir, self.ids[idx])
    mask_path = os.path.join(self.mask_dir, self.ids[idx])

    img = read_image(img_path).to(self.device).type(torch.float)
    mask = read_image(mask_path).to(self.device).type(torch.float)

    masks = [(mask == pixel_color) for pixel_color in self.label_map.values()]

    mask = torch.stack(masks, dim=-1)

    if self.transform:
      transform_out = self.transform(image=img, mask=mask)
      img, mask = transform_out["image"], transform_out["mask"]

    return img, mask

UNetBlock

In [ ]:
class UNetBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()
    self.conv_1 = Conv2dBatchNormReLU(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=3,
    )

    self.conv_2 = Conv2dBatchNormReLU(
        in_channels=out_channels,
        out_channels=out_channels,
        kernel_size=3,
    )

  def forward(self, x):
    out_conv_1 = self.conv_1(x)
    out_conv_2 = self.conv_2(out_conv_1)

    return out_conv_2


UNetEncoder

In [ ]:
class UNetEncoder(nn.Module):
  def __init__(
    self,
    in_channels=3
  ):
    super().__init__()

    self.block_1 = UNetBlock(
        in_channels=in_channels,
        out_channels=64,
    )
    self.max_pool_1 = nn.MaxPool2d(
        kernel_size=2,
        stride=2,
    )

    self.block_2 = UNetBlock(
        in_channels=64,
        out_channels=128,
    )
    self.max_pool_2 = nn.MaxPool2d(
        kernel_size=2,
        stride=2,
    )

    self.block_3 = UNetBlock(
        in_channels=128,
        out_channels=256
    )
    self.max_pool_3 = nn.MaxPool2d(
        kernel_size=2,
        stride=2,
    )

    self.block_4 = UNetBlock(
        in_channels=256,
        out_channels=512
    )


  def forward(self, x):
    out_block_1 = self.block_1(x)
    out_max_pool_1 = self.max_pool_1(out_block_1)

    out_block_2 = self.block_2(out_max_pool_1)
    out_max_pool_2 = self.max_pool_2(out_block_2)

    out_block_3 = self.block_3(out_max_pool_2)
    out_max_pool_3 = self.max_pool_3(out_block_3)

    out_block_4 = self.block_4(out_max_pool_3)

    return (out_block_4, out_block_3, out_block_2, out_block_1)

UNet center:

In [ ]:
class UNetCenter(nn.Module):
  def __init__(self):
    super().__init__()

    self.max_pool_1 = nn.MaxPool2d(
        kernel_size=2,
        stride=2,
    )

    self.block_1 = UNetBlock(
        in_channels=512,
        out_channels=1024,
    )

    self.up_conv_1 = nn.ConvTranspose2d(
        in_channels=1024,
        out_channels=512,
        kernel_size=2,
        stride=2,
    )

  def forward(self, x):
    out_max_pool_1 = self.max_pool_1(x)
    out_block_1 = self.block_1(out_max_pool_1)

    return self.up_conv_1(out_block_1)

UNet decoder

In [ ]:
class UNetDecoder(nn.Module):
  def __init__(self):
    super().__init__()

    self.block_1 = UNetBlock(
        in_channels=1024,
        out_channels=512
    )

    self.up_conv_1 = nn.ConvTranspose2d(
        in_channels=512,
        out_channels=256,
        kernel_size=2,
        stride=2,
    )

    self.block_2 = UNetBlock(
        in_channels=512,
        out_channels=256,
    )
    self.up_conv_2 = nn.ConvTranspose2d(
        in_channels=256,
        out_channels=128,
        kernel_size=2,
        stride=2,
    )

    self.block_3 = UNetBlock(
        in_channels=256,
        out_channels=128
    )
    self.up_conv_3 = nn.ConvTranspose2d(
        in_channels=128,
        out_channels=64,
        kernel_size=2,
        stride=2,
    )

    self.block_4 = UNetBlock(
        in_channels=128,
        out_channels=64
    )

  def forward(self, x, prev_feature_maps):
    feature_map_1 = crop(prev_feature_maps[0], top=0, left=0, height=x.shape[2], width=x.shape[3])
    concat_feature_map_1 = torch.cat((feature_map_1, x), dim=1)
    out_block_1 = self.block_1(concat_feature_map_1)

    out_up_conv_1 = self.up_conv_1(out_block_1)

    feature_map_2 = crop(prev_feature_maps[1], top=0, left=0, height=out_up_conv_1.shape[2], width=out_up_conv_1.shape[3])
    concat_feature_map_2 = torch.cat((feature_map_2, out_up_conv_1), dim=1)
    out_block_2 = self.block_2(concat_feature_map_2)

    out_up_conv_2 = self.up_conv_2(out_block_2)

    feature_map_3 = crop(prev_feature_maps[2], top=0, left=0, height=out_up_conv_2.shape[2], width=out_up_conv_2.shape[3])
    concat_feature_map_3 = torch.cat((feature_map_3, out_up_conv_2), dim=1)
    out_block_3 = self.block_3(concat_feature_map_3)

    out_up_conv_3 = self.up_conv_3(out_block_3)

    feature_map_4 = crop(prev_feature_maps[3], top=0, left=0, height=out_up_conv_3.shape[2], width=out_up_conv_3.shape[3])
    concat_feature_map_4 = torch.cat((feature_map_4, out_up_conv_3), dim=1)
    out_block_4 = self.block_4(concat_feature_map_4)

    return out_block_4

UNet tail

In [ ]:
class UNetTail(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.segmentation_tail = nn.Conv2d(
        in_channels=64,
        out_channels=num_classes,
        kernel_size=1
    )

  def forward(self, x):
    return self.segmentation_tail(x)

UNet

In [ ]:
class UNet(nn.Module):
  def __init__(self, in_channels, num_classes):
    super().__init__()

    self.enc = UNetEncoder(
        in_channels=in_channels
    )

    self.center = UNetCenter()

    self.dec = UNetDecoder()

    self.tail = UNetTail(
        num_classes=num_classes
    )

  def forward(self, x):
    out_enc = self.enc(x)
    out_center = self.center(out_enc[0])
    out_dec = self.dec(out_center, out_enc)
    out_tail = self.tail(out_dec)

    return out_tail

UNet preliminary testing

In [ ]:
# train_data = SegmentationDataset(
#     img_dir= "drive/MyDrive/CamVid/train",
#     mask_dir = "drive/MyDrive/CamVid/trainannot",
#     label_map = {
#         "sky": 0,
#         "road": 3,
#     },
#     transform=None,
#     device="cpu"
# )

In [ ]:
# fn_resize = Resize(size=(572, 572), antialias=True)
# x = fn_resize(train_data[0][0]).unsqueeze(0)
# x /= 255

# bench_unet = UNet(
#     in_channels=3,
#     num_of_classes=2
# )

# bench_unet(x).shape

torch.Size([1, 2, 388, 388])

In [ ]:
# train_data[0][0].shape

torch.Size([3, 360, 480])

In [ ]:
# bench_unet